# Bedrock with LangChain

## What is LangChain?

LangChain provides abstractions for:
- **Models** — Unified interface to different LLMs
- **Chains** — Sequences of operations
- **Memory** — Conversation history management
- **Agents** — Autonomous decision-making
- **Tools** — Integration with external services

## Installation

```bash
pip install langchain langchain-aws langchain-community
```

## ChatBedrock

In [ ]:
from langchain_aws import ChatBedrock
from langchain.schema import HumanMessage, AIMessage

# Initialize ChatBedrock
chat = ChatBedrock(
    model_id='anthropic.claude-3-sonnet-20240229-v1:0',
    region_name='us-east-1',
    model_kwargs={
        'temperature': 0.7,
        'max_tokens': 1024
    }
)

# Single message
response = chat.invoke([
    HumanMessage(content='What is AWS Bedrock?')
])
print(response.content)

# Multi-turn conversation
messages = [
    HumanMessage(content='What is machine learning?'),
    AIMessage(content='Machine learning is a subset of AI...'),
    HumanMessage(content='Give me an example')
]

response = chat.invoke(messages)
print(response.content)

## Embeddings

In [ ]:
from langchain_aws import BedrockEmbeddings

# Initialize embeddings
embeddings = BedrockEmbeddings(
    model_id='amazon.titan-embed-text-v2:0',
    region_name='us-east-1'
)

# Generate embedding for text
text = 'AWS Bedrock is a managed service'
embedding = embeddings.embed_query(text)
print(f'Embedding dimension: {len(embedding)}')

# Embed multiple texts
texts = [
    'AWS Bedrock',
    'Foundation models',
    'Machine learning'
]
embeddings_list = embeddings.embed_documents(texts)
print(f'Generated {len(embeddings_list)} embeddings')

## Chains

In [ ]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

# Create a prompt template
prompt = PromptTemplate(
    input_variables=['topic'],
    template='Explain {topic} in simple terms for beginners.'
)

# Create a chain
chain = LLMChain(llm=chat, prompt=prompt)

# Run the chain
result = chain.run(topic='Artificial Intelligence')
print(result)

## Conversation Memory

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# Create memory
memory = ConversationBufferMemory()

# Create conversation chain with memory
conversation = ConversationChain(
    llm=chat,
    memory=memory,
    verbose=True
)

# Multi-turn conversation
response1 = conversation.run(input='Hi, my name is Alice')
print(response1)

response2 = conversation.run(input='What is my name?')
print(response2)  # Model remembers "Alice"

# View conversation history
print(memory.buffer)

## RAG with LangChain

In [ ]:
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains import RetrievalQA

# Load documents
loader = TextLoader('company_docs.txt')
documents = loader.load()

# Split into chunks
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

# Create vector store
vector_store = FAISS.from_documents(chunks, embeddings)

# Create RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=chat,
    chain_type='stuff',
    retriever=vector_store.as_retriever(search_kwargs={'k': 5})
)

# Query
result = qa_chain.run('What is the vacation policy?')
print(result)

## Agents with LangChain

In [ ]:
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchRun

# Define tools
search = DuckDuckGoSearchRun()

tools = [
    Tool(
        name='Search',
        func=search.run,
        description='Useful for searching the internet'
    ),
    Tool(
        name='Calculator',
        func=lambda x: str(eval(x)),
        description='Useful for math calculations'
    )
]

# Create agent
agent = initialize_agent(
    tools,
    chat,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
result = agent.run('What is 2 + 2? Then search for AWS Bedrock pricing.')
print(result)

## Custom Tools

In [ ]:
from langchain.tools import tool
import boto3

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city"""
    # In real scenario, call weather API
    return f'Weather in {city}: Sunny, 72°F'

@tool
def query_database(query: str) -> str:
    """Query company database"""
    # In real scenario, query actual database
    return f'Query result: {query}'

# Use in agent
tools = [get_weather, query_database]

agent = initialize_agent(
    tools,
    chat,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

result = agent.run('What is the weather in New York?')
print(result)

## Streaming with LangChain

In [ ]:
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Create chat with streaming
chat_streaming = ChatBedrock(
    model_id='anthropic.claude-3-sonnet-20240229-v1:0',
    region_name='us-east-1',
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# Invoke with streaming
response = chat_streaming.invoke([
    HumanMessage(content='Write a 500-word essay on AI')
])

## Production Patterns

---

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary purpose of LangChain?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>To replace Bedrock</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>To provide abstractions for building LLM applications</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>To train custom models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>To filter harmful content</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What does ConversationBufferMemory do?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="0">
      <span>Stores embeddings</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="1">
      <span>Caches model responses</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="2">
      <span>Maintains conversation history for multi-turn interactions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="3">
      <span>Filters harmful content</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is a Chain in LangChain?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="0">
      <span>A sequence of operations combining prompts, models, and tools</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="1">
      <span>A blockchain for storing model outputs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="2">
      <span>A connection to multiple models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="3">
      <span>A security mechanism</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ How do you implement RAG with LangChain?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="0">
      <span>Use only the ChatBedrock class</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="1">
      <span>Combine embeddings, vector store, and RetrievalQA chain</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="2">
      <span>Use Bedrock Knowledge Bases only</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="3">
      <span>RAG is not supported in LangChain</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

In [ ]:
# ✅ Good: Error handling
from langchain.schema import LLMException

try:
    response = chat.invoke([HumanMessage(content=user_input)])
except LLMException as e:
    print(f'LLM error: {e}')
    return 'Sorry, I encountered an error.'

# ✅ Good: Input validation
def validate_input(text: str) -> bool:
    if len(text) > 10000:
        return False
    if len(text) < 1:
        return False
    return True

# ✅ Good: Response caching
from langchain.cache import InMemoryCache
import langchain

langchain.llm_cache = InMemoryCache()

# ✅ Good: Logging
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

logger.info(f'User input: {user_input}')
logger.info(f'Model response: {response}')